# Structured Outputs

LLMs regurgitate out text and that is great for so many applications. But in order to build strong, robust systems and applications, we need to make sense of the chaos sometimes by receiving a pre-determined structured output everytime an LLM is called.

## As always, libraries first!

In [22]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown


load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

# check if API keys are set
if not OPENAI_API_KEY:
    raise ValueError("Missing OpenAI API key")
if not GEMINI_API_KEY:
    raise ValueError("Missing Gemini API key")
if not ANTHROPIC_API_KEY:
    raise ValueError("Missing Anthropic API key")

## The Workflow

```mermaid
graph LR
    A[Generate Ticket] --> B[Respond to Ticket]
    B --> C[Evaluate Response]
    C --> B
    C --> D[Final Output]
```

## Creating Classes for LLMs responses

In [11]:
# classes
from pydantic import BaseModel

class CustomerReview(BaseModel):
    review: str
    rating: int

class ReviewResponse(BaseModel):
    response: str
    

class ResponseEvaluation(BaseModel):
    passed: bool
    feedback: str

## Calling Gemini to generate review tickets

In [25]:
# client
client = OpenAI()

In [34]:
# messages list
user_message = "I want you to generate a review for the home decore website. "
user_message += "The review should be a single sentence describing any product you can find in home decor. "
user_message += "Please ensure review has a rating - an int from 1 to 5 and it matches the tone of the review (positive review - 4 or 5, negative review - 1 or 2, neutral review - 3)."

messages = [{"role": "user", "content": user_message}]

In [35]:

structured_review = client.chat.completions.parse(
    model="gpt-4.1-nano",
    messages=messages,
    response_format=CustomerReview
)

structured_review = structured_review.choices[0].message.parsed
display(Markdown(f"### Structured Review:\n{structured_review}"))

### Structured Review:
review='The artisan-crafted ceramic vase adds a charming touch to my living room decor, and I absolutely love its vibrant colors.' rating=5

## Responding to the review

In [36]:
# messages list
message = "You are to answer the following customer review. Make sure to check the rating and answer accordingly (positive review - 4 or 5, negative review - 1 or 2, neutral review - 3).\n\n"
message += f"Review: {structured_review.review}\n"
message += f"Rating: {structured_review.rating}\n\n"

messages = [{"role": "user", "content": message}]

In [37]:
# structured response
review_response = client.chat.completions.parse(
    model="gpt-4.1-nano",
    messages=messages,
    response_format=ReviewResponse
)

review_response = review_response.choices[0].message.parsed
display(Markdown(f"### Response:\n{review_response.response}"))


### Response:
Thank you for your wonderful feedback! We're delighted to hear that you love the artisan-crafted ceramic vase and that it adds a charming touch to your living room. Your satisfaction means a lot to us. If you ever need more decor ideas or assistance, feel free to reach out!

## Lets evaluate our response

In [38]:
# messages list
message = "You are to evaluate the response for the following customer review. "
message += "You will determine if the proposed response is appropriate for the review and rating. "
message += f"Review: {structured_review.review}\n"
message += f"Rating: {structured_review.rating}\n\n"
message += f"Proposed Response: {review_response.response}\n"

messages = [{"role": "user", "content": message}]

In [39]:
# evaluate response
evaluator_response = client.chat.completions.parse(
    model="gpt-4.1-nano",
    messages=messages,
    response_format=ResponseEvaluation
)

evaluator_response = evaluator_response.choices[0].message.parsed
display(Markdown(f"### Passed:\n{evaluator_response.passed}"))
display(Markdown(f"### Feedback:\n{evaluator_response.feedback}"))

### Passed:
True

### Feedback:
The response appropriately acknowledges the customer's positive review, expresses gratitude, and reinforces their satisfaction. It also offers further assistance, which is good customer service. Overall, the response is suitable for a 5-star review.

<div style="border-radius:16px;background:#1e2a1e;margin:1em 0;padding:1em 1em 1em 3em;color:#eceff4;position:relative;box-shadow:0 6px 16px rgba(0,0,0,.4)">
  <b style="color:#a3be8c;font-size:1.25em">Your Challenge:</b>
  <ul style="margin:.6em 0 0;padding-left:1.2em;line-height:1.6">
    <li>Hey everyone! Ready to flex those agentic muscles? 🎉 Build a workflow just like the ticket system above, but for <b>product reviews</b>!</li>
    <li>Your workflow should:
      <ul>
        <li>Generate a product review (think: electronics, books, or your favorite kitchen gadget)</li>
        <li>Respond to the review (company reply, moderation, or a witty bot response)</li>
        <li>Evaluate the response (is it helpful, polite, and on point?)</li>
      </ul>
    </li>
    <li>Use structured outputs and Pydantic models for each step, just like we did above.</li>
    <li>Include an evaluator step to assess the quality of the response.</li>
    <li>Here’s a suggested workflow to get your creative gears turning:</li>
  </ul>
  <div style="position:absolute;top:-.8em;left:-.8em;width:2.4em;height:2.4em;border-radius:50%;background:#a3be8c;color:#2e3440;display:flex;align-items:center;justify-content:center;font-weight:700;font-size:1.2em">💪</div>
</div>

### Suggested Workflow

```mermaid
graph LR
    A[Generate Review] --> B[Respond to Review]
    B --> C[Evaluate Response]
    C --> B
    C --> D[Final Output]
```

Try to use structured outputs and Pydantic models for each step, just like in the notebook above. Include an evaluator step to assess the quality of the response.